In [21]:
import os
import time
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_cerebras import ChatCerebras
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
os.environ["CEREBRAS_API_KEY"] = os.getenv("CEREBRAS_API_KEY")
model = ChatCerebras(model="gpt-oss-120b")  # try this first

In [22]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

In [23]:
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 500),
            keep=("tokens", 200),
        ),
    ]
)

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [24]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

config = {"configurable": {"thread_id": "test-1"}}

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    print(f"\n {city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f" {response['messages'][-1].content}")
    print("-" * 50)
    time.sleep(2)


 Paris: ~741 tokens, 4 messages
 Here are a few options across different price ranges and styles to get you started. All of these properties are centrally located in Paris and have strong guest reviews.

| # | Hotel | Star rating | Approx. nightly price* | Key amenities | Neighborhood / Nearby attractions |
|---|-------|------------|------------------------|---------------|-----------------------------------|
| **1** | **Grand Hôtel Élysée** | 5 ★ | $350 | Spa, indoor pool, fully‑equipped gym, concierge, fine‑dining restaurant, 24 h room service | 8th arrondissement – steps from the Champs‑Élysées, Arc de Triomphe, and luxury shopping |
| **2** | **Hotel Le Marais** | 4 ★ | $185 | Business centre, meeting rooms, rooftop terrace, complimentary Wi‑Fi | 3rd arrondissement – heart of the historic Marais district, close to Musée Picasso and the Pompidou Centre |
| **3** | **City Inn Paris** | 4 ★ | $180 | Modern business centre, fitness room, on‑site café, free Wi‑Fi | 2nd arrondissement –

In [ ]:
#Human in loop middleware example
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langgraph.types import Command

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=MemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

config = {"configurable": {"thread_id": "test-approve"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
result

In [ ]:
if "__interrupt__" in result:
    print("Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f" Result: {result['messages'][-1].content}")

Paused! Approving...
 Result: The email has been sent to **john@test.com** with the subject “Hello” and the body “How are you?”. Let me know if there’s anything else you’d like to do!


### Reject

In [ ]:
agent_reject = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=MemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

config_reject = {"configurable": {"thread_id": "test-reject"}}

# Step 1: Request
result_reject = agent_reject.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config_reject
)

# Step 2: Reject
if "__interrupt__" in result_reject:
    print(" Paused! Rejecting...")
    
    result_reject = agent_reject.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config_reject
    )
    
    print(f" Result: {result_reject['messages'][-1].content}")

⏸️ Paused! Rejecting...
❌ Result: I’m not able to send the email right now because the request was cancelled. Would you like me to try sending it again, or is there something else I can help you with?


### Edit


In [ ]:
agent_edit = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=MemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

config_edit = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request with wrong info
result_edit = agent_edit.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config_edit
)

# Step 2: Edit and approve
if "__interrupt__" in result_edit:
    print(" Paused! Editing...")
    
    result_edit = agent_edit.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config_edit
    )
    
    print(f" Result: {result_edit['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: 
